# Making some trajectories for the frames to follow

In [1]:
%matplotlib widget

In [2]:
import numpy as np
import csv
import matplotlib.pyplot as plt
from numpy import sin, cos

## Helper functions

In [3]:
def retract_trajectory(trajectory, return_reverse=False):
    '''
    this simply takes a trajectory, reverses it, and concatenates it to the existing 
    trajectory. so if the original trajectory goes one way, the retracted trajectory
    will go that way and then come back. if you just want the reverse set return_reverse = True
    '''
    reverse = np.flip(trajectory,axis=0)
    if return_reverse:
        return reverse
    else:
        return np.vstack((trajectory,reverse))

In [4]:
def analyze_m_trajectory(traj, name=''):
    '''
    This gives some statistics on the values in a trajectory TODO raise alarms if it will self-collide
    '''
    m1_max = -25
    m1_min = -68.754
    m2_max = 1000       #not testing m2 max rn...
    m2_min = -5.38
    th_max = np.pi/4
    th_min = -np.pi/4

    traj_m1max= traj[:,0].max()
    traj_m1min = traj[:,0].min()
    traj_m2max= traj[:,1].max()
    traj_m2min = traj[:,1].min()
    traj_thmax= traj[:,2].max()
    traj_thmin = traj[:,2].min()

    # display
    print(f'*** {name} analysis ***')
    print(f'{name} m1 max = {traj_m1max}')
    print(f'{name} m1 min = {traj_m1min}')
    print(f'{name} m2 max = {traj_m2max}')
    print(f'{name} m2 min = {traj_m2min}')
    print(f'{name} th max = {traj_thmax}')
    print(f'{name} th min = {traj_thmin}')

    #check limits
    if traj_m1max > m1_max:
        print('M1 EXCEEDS MAXIMUM')
    if traj_m2max > m2_max:
        print('M2 EXCEEDS MAXIMUM')
    if traj_thmax > th_max:
        print('TH EXCEEDS MAXIMUM')
    if traj_m1min < m1_min:
        print('M1 EXCEEDS MINIMUM')
    if traj_m2min < m2_min:
        print('M2 EXCEEDS MINIMUM')
    if traj_thmin < th_min:
        print('TH EXCEEDS MINIMUM')

### Conversions

In [5]:
def r_to_m(r_traj,s=0.5,to_mm=True):
    '''
    converts an r-trajectory [Nx3] to an m-trajectory [Nx3]
    '''
    delta = np.array([-1.5875,15.8750,0]) #mm values
    if to_mm:
        fs = 63.5
    else:
        delta = delta/fs #convert to normalized
        fs = 1

    m_traj = np.zeros(r_traj.shape)
    for i,r_i in enumerate(r_traj):
        r1, r2, th = r_i
        
        #load converted values
        m_traj[i] = np.array([fs*r1 + delta[0]*cos(th) - delta[1]*sin(th),
                              fs*r2 + delta[0]*sin(th) + delta[1]*cos(th),
                              th]) #spelling it out because somethings wrong

    return m_traj

In [6]:
def p_to_r(p_traj,s=0.5):
    '''
    converts a p-trajectory [Nx3] to an r-trajectory [Nx3]
    this doesnt check sizes so make sure its not 1-dimensional
    '''
    r_traj = np.zeros(p_traj.shape)
    for i,p_i in enumerate(p_traj):
        p1, p2, th = p_i
        w1 = s*sin(th)
        w2 = 1-s*cos(th)
        r_over_p = np.array([[w2, w1, 0],
                             [-w1,w2, 0],
                             [0,0,1]])
        
        #load converted values
        r_traj[i] = r_over_p @ p_traj[i]
    
    return r_traj

In [7]:
def p_to_m(p_traj,s=0.5):
    '''
    converts a p-trajectory [Nx3] to an m-trajectory [Nx3]
    '''
    return r_to_m(p_to_r(p_traj))

In [8]:
def m_to_r(m_traj,to_normalized=True):
    '''
    converts an m-trajectory [Nx3] to an r-trajectory [Nx3]
    '''
    delta = np.array([1.5875,15.8750,0]) #mm values
    if to_normalized:
        fs = 63.5
    else:
        delta = delta/fs #convert to normalized
        fs = 1

    r_traj = np.zeros(m_traj.shape)
    for i,m_i in enumerate(m_traj):
        m1, m2, th = m_i
        
        #load converted values
        r_traj[i] = np.array([m1 - delta[0]*cos(th) + delta[1]*sin(th),
                              m2 - delta[0]*sin(th) - delta[1]*cos(th),
                              th])/fs

    return r_traj

### Saving Trajectories

In [9]:
def traj_to_csv(traj,dt,filename):
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['dt', dt])
        writer.writerow(['r1', 'r2', 'theta'])
        for r1,r2,theta in traj:
            writer.writerow([r1,r2,theta])
    
    print('wrote ', filename)

## Generate Trajectories

In [10]:
'''
Trajectory 1: motion in r1
r1 = -.75 to -0.3
r2 = 0.25
th = 0
'''
Npts = 80
traj1_r1 = np.linspace(-.8,-0.4,num=Npts)
traj1_r2 = 0.1*np.ones(Npts)
traj1_th = np.zeros(Npts)

traj1_r = np.vstack((traj1_r1,traj1_r2,traj1_th)).T
traj1_m = r_to_m(traj1_r)

analyze_m_trajectory(traj1_m, name='traj1')

'''
Traj 2: motion in r1 with theta nonzero
'''
Npts = 80
th_value = 0.5
traj2_r1 = np.linspace(-.75,-0.3,num=Npts)
traj2_r2 = 0.1*np.ones(Npts)
traj2_th = th_value*np.ones(Npts)

traj2_r = np.vstack((traj2_r1,traj2_r2,traj2_th)).T
traj2_m = r_to_m(traj2_r)

analyze_m_trajectory(traj2_m, name='traj2')

'''
Trajectory 3: motion in r2
r1 = -0.5
r2 = -0.2 to 0.7
th = 0
'''
Npts = 80
traj3_r1 = -0.5*np.ones(Npts)
traj3_r2 = np.linspace(-0.2,0.7,num=Npts)
traj3_th = np.zeros(Npts)

traj3_r = np.vstack((traj3_r1,traj3_r2,traj3_th)).T
traj3_m = r_to_m(traj3_r)

analyze_m_trajectory(traj3_m, name='traj3')

'''
Traj 4: motion in r2 with theta nonzero
'''
Npts = 80
th_value = 0.5
traj4_r1 = -0.5*np.ones(Npts)
traj4_r2 = np.linspace(-0.2,0.5,num=Npts)
traj4_th = th_value*np.ones(Npts)

traj4_r = np.vstack((traj4_r1,traj4_r2,traj4_th)).T
traj4_m = r_to_m(traj4_r)

analyze_m_trajectory(traj4_m, name='traj4')

'''
Trajectory 5: motion in theta
r1 = -0.6,
r2 = 0.
th = 0. to -0.04 radians.
'''
Npts = 80
traj5_r1 = -0.5*np.ones(Npts)
traj5_r2 = 0.0*np.ones(Npts)
traj5_th = np.linspace(-0.3,0.7,num=Npts)

traj5_r = np.vstack((traj5_r1,traj5_r2,traj5_th)).T
traj5_m = r_to_m(traj5_r)

analyze_m_trajectory(traj5_m, name='traj5')

'''
Trajectory 6: Motion around the midpoint.
m1 = -0.75*fs
m2 = 0.5*fs
th = 0.1 to -0.04 radians.
'''
fs = 63.5 #mm
Npts = 80
traj6_m1 = -0.75*fs*np.ones(Npts)
traj6_m2 = 0.5*fs*np.ones(Npts)
traj6_th = np.linspace(-0.7,0.7,num=Npts)

traj6_m = np.vstack((traj6_m1,traj6_m2,traj6_th)).T

analyze_m_trajectory(traj6_m, name='traj6')

'''
Trajectory 7: “1 DOF” motion. 
Bottom protraction = -pi/7, 
top protraction = pi/6. 
theta from -pi/6 to pi/6
TODO
'''

'''
Trajectory 8: motion in p1
p1 = -1.6 to -0.66
p2 = 0.81
th = -0.42
'''
Npts = 80
traj8_p1 = np.linspace(-1.6,-0.66,num=Npts)
traj8_p2 = 0.81*np.ones(Npts)
traj8_th = -0.42*np.ones(Npts)

traj8_p = np.vstack((traj8_p1,traj8_p2,traj8_th)).T
traj8_m = p_to_m(traj8_p)

analyze_m_trajectory(traj8_m, name='traj8')

'''
Trajectory 9: motion in p2
p1 = -0.83
p2 from 0.5 to 1.3
theta = -0.42
'''
Npts = 80
traj9_p1 = -0.83*np.ones(Npts)
traj9_p2 = np.linspace(0.4,1.3,num=Npts)
traj9_th = -0.42*np.ones(Npts)

traj9_p = np.vstack((traj9_p1,traj9_p2,traj9_th)).T
traj9_m = p_to_m(traj9_p)

analyze_m_trajectory(traj9_m, name='traj9')

'''
Trajectory 10: Motion in theta, constant p. p1 = -1, p2 = 0.5, theta moved from -0.6 to 0.6
p1 = -1
p2 = 0.5
th = -0.6 to 0.6
'''
Npts = 80
traj10_p1 = -1.2*np.ones(Npts)
traj10_p2 = 0.5*np.ones(Npts)
traj10_th = np.linspace(-0.45,0.45,num=Npts)

traj10_p = np.vstack((traj10_p1,traj10_p2,traj10_th)).T
traj10_m = p_to_m(traj10_p)

analyze_m_trajectory(traj10_m, name='traj10')

*** traj1 analysis ***
traj1 m1 max = -26.9875
traj1 m1 min = -52.3875
traj1 m2 max = 22.225
traj1 m2 min = 22.225
traj1 th max = 0.0
traj1 th min = 0.0
*** traj2 analysis ***
traj2 m1 max = -28.05404274234269
traj2 m1 min = -56.62904274234269
traj2 m2 max = 19.520535127475494
traj2 m2 min = 19.520535127475494
traj2 th max = 0.5
traj2 th min = 0.5
*** traj3 analysis ***
traj3 m1 max = -33.3375
traj3 m1 min = -33.3375
traj3 m2 max = 60.324999999999996
traj3 m2 min = 3.174999999999999
traj3 th max = 0.0
traj3 th min = 0.0
*** traj4 analysis ***
traj4 m1 max = -40.75404274234269
traj4 m1 min = -40.75404274234269
traj4 m2 max = 44.920535127475496
traj4 m2 min = 0.4705351274754932
traj4 th max = 0.5
traj4 th min = 0.5
*** traj5 analysis ***
traj5 m1 max = -28.575213395738132
traj5 m1 min = -43.19114275721247
traj5 m2 max = 15.954138928972924
traj5 m2 min = 11.11917414465142
traj5 th max = 0.7
traj5 th min = -0.3
*** traj6 analysis ***
traj6 m1 max = -47.625
traj6 m1 min = -47.625
traj6 m2 m

## Save trajectories

In [11]:
#save the trajectories
dt = .05 
traj_to_csv(traj1_m,dt,'traj1.csv')
traj_to_csv(traj2_m,dt,'traj2.csv')
traj_to_csv(traj3_m,dt,'traj3.csv')
traj_to_csv(traj4_m,dt,'traj4.csv')
traj_to_csv(traj5_m,dt,'traj5.csv')
traj_to_csv(traj6_m,dt,'traj6.csv')
traj_to_csv(traj8_m,dt,'traj8.csv')
traj_to_csv(traj9_m,dt,'traj9.csv')
traj_to_csv(traj10_m,dt,'traj10.csv')

wrote  traj1.csv
wrote  traj2.csv
wrote  traj3.csv
wrote  traj4.csv
wrote  traj5.csv
wrote  traj6.csv
wrote  traj8.csv
wrote  traj9.csv
wrote  traj10.csv


In [12]:
#save retracted trajectories
dt = .05 
traj_to_csv(retract_trajectory(traj1_m),dt,'traj1_retracted.csv')
traj_to_csv(retract_trajectory(traj2_m),dt,'traj2_retracted.csv')
traj_to_csv(retract_trajectory(traj3_m),dt,'traj3_retracted.csv')
traj_to_csv(retract_trajectory(traj4_m),dt,'traj4_retracted.csv')
traj_to_csv(retract_trajectory(traj5_m),dt,'traj5_retracted.csv')
traj_to_csv(retract_trajectory(traj6_m),dt,'traj6_retracted.csv')
traj_to_csv(retract_trajectory(traj8_m),dt,'traj8_retracted.csv')
traj_to_csv(retract_trajectory(traj9_m),dt,'traj9_retracted.csv')
traj_to_csv(retract_trajectory(traj10_m),dt,'traj10_retracted.csv')

wrote  traj1_retracted.csv
wrote  traj2_retracted.csv
wrote  traj3_retracted.csv
wrote  traj4_retracted.csv
wrote  traj5_retracted.csv
wrote  traj6_retracted.csv
wrote  traj8_retracted.csv
wrote  traj9_retracted.csv
wrote  traj10_retracted.csv
